# Run the black-box judge on a free Colab GPU with a Hugging Face model

No API key and no quota. The model is pulled from Hugging Face and served locally with Ollama.

1. Runtime menu, Change runtime type, T4 GPU.
2. In the repo root run `python scripts/make_judge_bundle.py --with-labels` and upload `judge_bundle.zip` in cell 3.
3. Run all cells. Results download as `judge_results.zip`; unzip it into the repo root so the files land in `results/tables/`.

The run resumes if the session drops: upload the zip again and rerun. Use the same judge model for every conversation set.

In [ ]:
import subprocess, time
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

In [ ]:
# A Hugging Face GGUF repo. Use a different family from the model under test (not Qwen for Qwen).
JUDGE_MODEL = "hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_K_M"
!ollama pull {JUDGE_MODEL}

In [ ]:
import os, zipfile
from google.colab import files
up = files.upload()
for name in up:
    zipfile.ZipFile(name).extractall(".")
!pip -q install pyyaml tqdm
os.environ["OLLAMA_API_KEY"] = "local"

In [ ]:
ENDPOINT = "http://localhost:11434/v1/chat/completions"
# (data tag, output name). Tag "" is Qwen; the others need data/multi_turn_gen_<tag>/ in the zip.
RUNS = [("", "qwen2.5-3b"), ("smollm2-1.7b", "smollm2-1.7b"), ("olmo-2-1b", "olmo-2-1b")]
for tag, name in RUNS:
    folder = f"data/multi_turn_gen_{tag}" if tag else "data/multi_turn_gen"
    if not os.path.isdir(folder):
        print("skip", name, "(no", folder + ")"); continue
    tag_arg = f"--data-tag {tag}" if tag else ""
    !PYTHONPATH=src python scripts/run_judge.py {tag_arg} --judge-model {JUDGE_MODEL} --endpoint {ENDPOINT} --api-key-env OLLAMA_API_KEY --min-interval 0 --out results/tables/judge_hf_{name}.jsonl

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob("results/tables/judge_hf_*.jsonl")):
    d = pd.read_json(f, lines=True)
    print(f, len(d), d.status.value_counts().to_dict())
!zip -qr judge_results.zip results/tables/judge_hf_*.jsonl
files.download("judge_results.zip")